In [1]:
# Load Titanic dataset
import seaborn as sns
titanic = sns.load_dataset("titanic")

In [2]:
import pandas as pd

from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate

import json

Manual data exploration:

In [3]:
titanic.columns

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone'],
      dtype='object')

In [4]:
titanic.shape

(891, 15)

In [5]:
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [6]:
titanic.agg({'survived':'mean'})

survived    0.383838
dtype: float64

In [7]:
titanic.groupby('sex')['survived'].mean().to_string()

'sex\nfemale    0.742038\nmale      0.188908'

In [8]:
len(titanic[titanic['sex']=='male'])

577

In [21]:
len(titanic[titanic['pclass']==1])

216

Creating functions that our AI model will use to explore Titanic data:

In [9]:
@tool
def get_columns() -> str:
    """Return the column names of the active dataframe."""
    return str(list(df.columns))

@tool
def group_mean(group_col, target_col) -> str:
    """Groups the dataset, computes the mean of target_col, and returns the result as a string."""
    return df.groupby(group_col)[target_col].mean().to_string()

@tool
def count_where(column, value) -> int:
    """Returns how many rows match a condition."""
    return len(df[df[column]==value])

In [10]:
tools = [get_columns, group_mean, count_where]

In [11]:
# Build prompt
prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a data analyst assistant.\n"
     "Use tools when needed to answer questions about the dataset.\n"
     "Pay close attention to the column names you are using. They are case sensitive.\n"
     "Be concise and explain results clearly."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

In [12]:
# Reading in OpenAI api key
with open('../keys.json') as fi:
    credentials = json.load(fi)

api_key = credentials['api_key']

In [13]:
# Creating a ChatOpenAI instance
llm = ChatOpenAI(
    model_name="openrouter/owl-alpha",
    base_url="https://openrouter.ai/api/v1", 
    openai_api_key=api_key
)

In [14]:
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [15]:
# Identify the DataFrame that the agent should reference when using the tools
df = sns.load_dataset("titanic")

In [16]:
agent_executor.invoke({"input": "Did women survive more often than men?"})



> Entering new AgentExecutor chain...

Invoking: `get_columns` with `{}`
responded: I'll help you analyze the survival rates by gender. Let me first check the available columns in the dataset to understand the structure.

1;3m['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']
Invoking: `group_mean` with `{'group_col': 'sex', 'target_col': 'survived'}`


sex
female    0.742038
Yes, women survived more often than men. The survival rates were:

- **Females**: 74.2% survival rate (0.742038)
- **Males**: 18.9% survival rate (0.188908)

Women had a much higher survival rate than men - about 4 times higher.

> Finished chain.


{'input': 'Did women survive more often than men?',
 'output': 'Yes, women survived more often than men. The survival rates were:\n\n- **Females**: 74.2% survival rate (0.742038)\n- **Males**: 18.9% survival rate (0.188908)\n\nWomen had a much higher survival rate than men - about 4 times higher.'}

In [17]:
agent_executor.invoke({"input": "Which passenger class had the highest survival rate?"})



> Entering new AgentExecutor chain...

Invoking: `get_columns` with `{}`


1;3m['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']
Invoking: `group_mean` with `{'group_col': 'pclass', 'target_col': 'survived'}`


pclass
1    0.629630
2    0.472826
**First class (pclass = 1)** had the highest survival rate at **62.96%**.

Here's the breakdown by passenger class:

| Passenger Class | Survival Rate |
|-----------------|---------------|
| 1st Class      | 62.96%        |
| 2nd Class      | 47.28%        |
| 3rd Class      | 24.24%        |

The survival rate decreases significantly with each lower class, with 1st class passengers being about **2.6 times** more likely to survive than 3rd class passengers.

> Finished chain.


{'input': 'Which passenger class had the highest survival rate?',
 'output': "**First class (pclass = 1)** had the highest survival rate at **62.96%**.\n\nHere's the breakdown by passenger class:\n\n| Passenger Class | Survival Rate |\n|-----------------|---------------|\n| 1st Class      | 62.96%        |\n| 2nd Class      | 47.28%        |\n| 3rd Class      | 24.24%        |\n\nThe survival rate decreases significantly with each lower class, with 1st class passengers being about **2.6 times** more likely to survive than 3rd class passengers."}

In [20]:
agent_executor.invoke({"input": "How many first class passengers were there?"})



> Entering new AgentExecutor chain...

Invoking: `get_columns` with `{}`


1;3m['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alive', 'alone']
Invoking: `count_where` with `{'column': 'pclass', 'value': 1.0}`


There were **216** first-class passengers in the dataset.

> Finished chain.


{'input': 'How many first class passengers were there?',
 'output': 'There were **216** first-class passengers in the dataset.'}